# # 04 · Anatomy: registration and the Allen atlas

Companion to the tutorial notebook [`04_anatomy_allen`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level2).

**Kernel:** `cajal-lipidomics` · **Reads:** `03_normalized.h5ad` · **Writes:** `results/region_by_lipid.csv` (no new pipeline stage)

A lipid map without anatomy is a pretty picture. With an anatomical label on every pixel it becomes neuroscience: we can compare control and pregnant **region by region**, and later join our lipids to gene expression in the same regions. This notebook is about how every pixel gets that address.

**The plan**
1. What a reference atlas is
2. Registration = an affine transform + a nonlinear warp (toy demos)
3. ABBA, the one piece shipped pre-computed
4. From CCF coordinate to voxel to region, and reading the slicing angle off the data
5. Region maps and region outlines
6. Grey vs white matter, and does a myelin lipid agree?
7. The region x lipid matrix that powers later notebooks

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------
from lipid_project import paths
paths.ensure_dirs()    # creates data/derived, results/ and figures/ if they're missing
paths.summary()    # prints where every path points, handy to spot a wrong tutorial location

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

import cajal_lipidomics as cl
from cajal_lipidomics import plotting
from cajal_lipidomics.style import FS
cl.style.set_style()    # the tutorial's matplotlib style, so my figures match theirs

print("ready. cajal_lipidomics", cl.__version__)

# ## 1. What a reference atlas is

The scanner only knows that a pixel was at row `y`, column `x` of the slide. That's meaningless across sections: tissue gets placed at a different angle, cut at a slightly different depth, brains differ in size. We need a **shared coordinate system** every section can be mapped into.

The **Allen Mouse Brain Common Coordinate Framework v3 (CCFv3)** is the standard one. Two things bolted together:
- a **3D coordinate space** of 25 µm voxels: any point is `(x, y, z)` in mm along the anterior-posterior (AP), dorsal-ventral and medial-lateral axes. Think latitude, longitude, altitude.
- an **annotation volume**: experts labelled every voxel with one of ~1000 regions, organized as a tree (cortex → somatosensory cortex → layer 2/3...). Each region has an acronym (`CP` = caudoputamen) and a colour.

Once a pixel has a CCF coordinate, its region is a plain lookup. Getting the coordinate is **registration**.

# ## 2. Registration: affine + nonlinear warp

**Layer 1, the affine transform**, fixes the gross misalignment: rotation, scaling (including uneven shrinkage along one axis), shear and a shift. In 2D a point `p` moves to `p' = A p + t`, with `A` a 2x2 matrix and `t` a vector. Affine maps keep straight lines straight and parallel lines parallel.

In [ ]:
# A 6x6 grid of points standing in for landmarks on a section
gx, gy = np.meshgrid(np.linspace(0, 1, 6), np.linspace(0, 1, 6))    # two 6x6 arrays: the x and y coordinate of every grid point
pts = np.column_stack([gx.ravel(), gy.ravel()])                  # (36 points, 2 coords)

theta = np.deg2rad(20)                                            # rotate by 20 degrees
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])                   # the standard 2D rotation matrix
Sc = np.diag([1.3, 0.7])                                          # stretch x by 1.3, squash y to 0.7
A = R @ Sc                                                        # matrix product: scale first, then rotate
t = np.array([0.4, -0.2])                                         # then shift

# Apply A p + t to every point at once. Points are rows, so p' = p @ A.T + t.
pts_affine = pts @ A.T + t

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(*pts.T, s=18, c="0.6", label="original grid (section)")
ax.scatter(*pts_affine.T, s=18, c="crimson", label="after A·p + t")
ax.set_aspect("equal"); ax.legend()
ax.set_title("affine: rotate, scale, shift (lines stay straight)")
plt.tight_layout()
plt.show()
print("A =\n", A.round(2), "\nt =", t)

# Real tissue also tears, folds and swells locally. One matrix acts on the whole plane identically, so it can't fix a fold in one corner. **Layer 2, a nonlinear warp**, is a smooth displacement field: a little arrow at every location saying "move this bit of tissue this much, this way". Good methods keep the field smooth and invertible (a *diffeomorphism*: no tearing, no folding over). ABBA uses elastix splines; STalign uses LDDMM.

In [ ]:
x_, y_ = pts_affine[:, 0], pts_affine[:, 1]    # start from the affine result and add a wiggle on top
amp = 0.12                                         # how strong the bend is
dx = amp * np.sin(2 * np.pi * y_)                  # horizontal push varies smoothly with y
dy = amp * np.sin(2 * np.pi * x_)                  # vertical push varies smoothly with x
pts_warped = np.column_stack([x_ + dx, y_ + dy])    # glue the shifted x and y back into (36, 2)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(*pts_affine.T, s=18, c="crimson", label="affine only")
ax.scatter(*pts_warped.T, s=22, c="navy", label="affine + nonlinear warp")
# arrows show the displacement field: where each point got pushed
ax.quiver(x_, y_, dx, dy, angles="xy", scale_units="xy", scale=1, width=0.004, color="0.5")
ax.set_aspect("equal"); ax.legend()
ax.set_title("nonlinear warp: a smooth displacement field (lines bend)")
plt.tight_layout()
plt.show()

# ## 3. ABBA, the one piece shipped pre-computed

The community tool for registering brain sections to CCFv3 is **ABBA** (Aligning Big Brains and Atlases), a Fiji/QuPath plugin from the EPFL bioimaging platform: a machine-learning pre-alignment (DeepSlice) guesses the atlas level and cutting angle, an affine elastix step, a spline elastix warp, and manual landmarks in BigWarp for stubborn slices. It also corrects the **slicing angle** (a coronal block is rarely cut perfectly perpendicular to the AP axis).

It's shipped pre-computed because it's Java/C++, GUI-driven, and setup rather than science. Its output, a CCF coordinate and region per pixel, is what notebook 01 already attached to `obs`.

# ## 4. From coordinate to voxel to region

We switch `X` to the uMAIA-normalized values once, up front, so every lipid view below uses them.

In [ ]:
adata = ad.read_h5ad(paths.stage("normalized"))    # the output of notebook 03
adata.X = adata.layers["umaia"]          # in memory only: the file on disk keeps raw X + the umaia layer
names = adata.var["lipid"].astype(str).to_list()    # human-readable lipid names (var_names are formulas)
print("pixels x lipids:", adata.shape)
print("pixels per section:", adata.obs["SectionID"].value_counts().to_dict())
adata.obs[["Condition", "xccf", "yccf", "zccf", "acronym", "allencolor"]].head()

# CCF voxels are 25 µm, so 1 mm = 40 voxels. A coordinate in mm becomes a voxel index by multiplying by 40 and flooring (any point inside a voxel belongs to it): **index = ⌊ccf_mm × 40⌋**. That integer triple is the address you'd read in the annotation volume.

In [ ]:
# CCF coords are in mm; the atlas grid is 25 um voxels, so voxel = floor(mm * 40)
for axis in ["xccf", "yccf", "zccf"]:
    # xccf -> x_index etc.; integer voxel indices I can use to look things up in the atlas volume
    adata.obs[axis.replace("ccf", "_index")] = np.floor(adata.obs[axis].to_numpy() * 40).astype(int)
adata.obs[["xccf", "x_index", "yccf", "y_index", "zccf", "z_index", "acronym"]].head(4)

# **Reading the slicing angle off the data.** Here `xccf` is the AP axis. A perfectly coronal cut sits at one AP level, so its `x_index` takes one value. A tilted cut grazes a range of AP levels across its width.

In [ ]:
# sanity check: each section should sit at (roughly) one anterior-posterior level
for c in ["naive", "pregnant"]:
    sub = adata.obs[adata.obs["Condition"] == c]
    print(f"{c:9s}: {sub['x_index'].nunique():2d} distinct AP level(s), "
          f"xccf from {sub['xccf'].min():.3f} to {sub['xccf'].max():.3f} mm")

# If one section spans several AP levels, ABBA recognised a tilted cut and assigned each part of the section to the level it truly belongs to, instead of forcing it onto one plane.

## 5. Region maps and outlines

In [ ]:
axes = plotting.spatial_categorical(adata, color_key="allencolor", point_size=3.0)    # colour every pixel by its Allen region's official colour
axes[0].figure.suptitle("Allen regions on each section", y=1.02, fontsize=FS["l"])
plt.show()

print("distinct Allen regions:", adata.obs["acronym"].nunique())
print("\nlargest regions by pixel count:")
print(adata.obs["acronym"].value_counts().head(10).to_string())

# Both sections show the same regions in the same places because both were warped into the same atlas, even though the raw slides aren't pixel-for-pixel identical.

A filled map with dozens of colours is busy. Usually we want just the **outlines**, drawn thin over a lipid map. The rule is local: **a pixel is on a boundary if a neighbour belongs to a different region.** Toy first.

In [ ]:
toy = np.array([[0, 0, 0, 1, 1, 1],
                [0, 0, 0, 1, 1, 1],
                [0, 0, 2, 2, 1, 1],
                [0, 2, 2, 2, 2, 1],
                [2, 2, 2, 2, 2, 2],
                [2, 2, 2, 2, 2, 2]])
edge = np.zeros_like(toy, bool)    # same shape as toy, all False to start
edge[:, :-1] |= toy[:, :-1] != toy[:, 1:]       # differs from its right-hand neighbour
edge[:-1, :] |= toy[:-1, :] != toy[1:, :]       # differs from the neighbour below

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(toy, cmap="Pastel1")
ey, ex = np.nonzero(edge)    # row/col indices of the boundary pixels (rows = y, cols = x)
ax.scatter(ex, ey, s=160, facecolors="none", edgecolors="k", lw=1.6)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("circled = sits on a boundary")
plt.tight_layout()
plt.show()

# `plotting.allen_contours` runs exactly those two comparisons on the real pixel grid. We draw one grey-matter phospholipid on the control section with the Allen outlines on top: if anatomy and chemistry agree, the lines land on the edges of the colour.

In [ ]:
# Pick a PC (phosphatidylcholine, grey-matter leaning). Fall back to the most variable lipid.
pc_cols = [i for i, n in enumerate(names) if n.startswith("PC ")]    # column indices of all PC species
X = np.asarray(adata.X)    # plain numpy array, easier to slice than the AnnData view
# pick the PC with the highest coefficient of variation (std / mean), i.e. the most spatially patterned one
jc = max(pc_cols, key=lambda i: X[:, i].std() / (X[:, i].mean() + 1e-9)) if pc_cols else int(np.argmax(X.std(0)))

ctrl_sid = adata.obs.loc[adata.obs["Condition"] == "naive", "SectionID"].iloc[0]    # the first control section
m = (adata.obs["SectionID"] == ctrl_sid).to_numpy()    # boolean mask: pixels on that section
sub = adata.obs[m]
v = X[m, jc]    # that lipid's values on that section

fig, ax = plt.subplots(figsize=(6.5, 5.2))
# y flipped so the section isn't upside down; colour limits at the 2nd/98th percentile so a few hot pixels don't wash out the rest
ax.scatter(sub["x"], -sub["y"], c=v, cmap="plasma", s=5, rasterized=True,
           vmin=np.quantile(v, 0.02), vmax=np.quantile(v, 0.98))
plotting.allen_contours(sub, ax, color="k", lw=0.3, s=0.6)     # outlines in pixel coordinates (x, -y)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"{names[jc]} with Allen region outlines (control)")
plt.tight_layout()
plt.show()

# ## 6. Grey vs white matter

The biggest split in brain lipid composition: **white matter** is packed with myelin (sphingolipid-rich), **grey matter** with cell bodies and synapses (phospholipid-rich). If a method can't see this, it can't see anything.

The Allen ontology groups every white-matter structure under "fiber tracts", and the Allen palette paints that whole subtree one grey, **`#cccccc`**. So a pixel is white matter exactly when its `allencolor` is `#cccccc`: the atlas's own definition, no hand-typed acronym list. We print which regions it picked so the choice is auditable.

In [ ]:
# the Allen atlas paints all fibre tracts this light grey, so the colour itself is the label
WHITE_MATTER_COLOR = "#cccccc"
is_wm = adata.obs["allencolor"].astype(str).str.lower().eq(WHITE_MATTER_COLOR)    # True for white-matter pixels; lower() in case some hexes are uppercase
adata.obs["matter"] = np.where(is_wm, "white", "grey")    # a simple 2-level label I reuse in notebooks 07 and 09

print("regions selected as white matter:", sorted(adata.obs.loc[is_wm, "acronym"].astype(str).unique()))
print(adata.obs["matter"].value_counts().to_string())
print(f"white matter = {100 * is_wm.mean():.1f}% of pixels")

# spatial_categorical wants a column of hex colours
adata.obs["matter_color"] = adata.obs["matter"].map({"white": "#111111", "grey": "#cfcfcf"})
axes = plotting.spatial_categorical(adata, color_key="matter_color", point_size=3.0)
axes[0].figure.suptitle("white matter (black) vs grey matter (light), from the atlas alone", y=1.02, fontsize=FS["l"])
plt.show()

# ### Does a myelin lipid agree with the anatomy?

If white matter is myelin and myelin is sphingolipid, then a myelin marker like `HexCer 42:2` should be far brighter in the white-matter pixels. The atlas never saw the lipids, the lipids never saw the atlas. If they agree, that's the first real proof the registration is sound.

In [ ]:
# HexCer 42:2 is the classic myelin lipid; if it's not in the data, the `or` falls back to any sphingolipid
hex_cols = [i for i, n in enumerate(names) if n.startswith("HexCer 42:2")] or \
           [i for i, n in enumerate(names) if n.startswith(("HexCer", "SM", "Cer"))]
j = hex_cols[0]    # just take the first match
v = X[:, j]
wm_mask = is_wm.to_numpy()    # numpy boolean mask, so I can index arrays with it
print(f"{names[j]}: mean in white matter {v[wm_mask].mean():.4f} vs grey {v[~wm_mask].mean():.4f}  "
      f"-> {v[wm_mask].mean() / v[~wm_mask].mean():.1f}x")

plotting.spatial_lipid(adata, adata.var_names[j], point_size=3.0)   # var_names are formulas
plt.suptitle(f"{names[j]}: the myelin lipid traces the tracts", y=1.02, fontsize=FS["m"])
plt.show()

# One lipid is an anecdote. Which lipids, out of all of them, are most enriched in white matter? For each lipid: log2(mean in white matter / mean everywhere else), sorted. Sphingolipid classes in red.

In [ ]:
adw = ad.AnnData(X, obs=adata.obs.copy())    # a lightweight copy whose var_names are lipid names, so the bar labels are readable
adw.var_names = ad.utils.make_index_unique(pd.Index(names))          # label bars by lipid name
SPH = ("HexCer", "SHexCer", "SM", "Cer", "CerP")    # sphingolipid class prefixes; str.startswith accepts a tuple
# red for sphingolipids, blue for everything else
props = {"color": {n: ("#cc3311" if n.startswith(SPH) else "#4477aa") for n in adw.var_names}}
plotting.marker_barplot(adw, wm_mask, lipid_props=props, top=min(20, adata.n_vars),
                        title="lipids enriched in white matter vs the rest (red = sphingolipid)")
plt.tight_layout()
plt.show()

# The barplot knew nothing about myelin; it compared two sets of pixels. Sphingolipids rising to the top is exactly what biochemistry predicts.

## 7. The region x lipid matrix

One row per Allen region, one column per lipid, each cell the mean intensity. It's just a `groupby` mean, and it's the table that powers region-level differential tests and the gene join later. We save it to `results/` too.

In [ ]:
lipid_df = pd.DataFrame(X, columns=adw.var_names)    # one row per pixel, one column per lipid
lipid_df["acronym"] = adata.obs["acronym"].astype(str).to_numpy()    # attach each pixel's region; to_numpy avoids index misalignment
region_by_lipid = lipid_df.groupby("acronym").mean()    # average every lipid within each region: the region x lipid matrix
print("region x lipid matrix:", region_by_lipid.shape)

region_by_lipid.to_csv(paths.RESULTS / "region_by_lipid.csv")
print("saved", paths.RESULTS / "region_by_lipid.csv")
region_by_lipid.iloc[:5, :4]

# `sorted_lipid_heatmap` shows the same table as **relative abundance** (each lipid's region mean divided by its overall mean, so 1 = average, 2 = double) and orders both axes by **hierarchical clustering on cosine distance**, so similar regions sit together and co-varying lipids form blocks.

In [ ]:
ax, sorted_df = plotting.sorted_lipid_heatmap(adata, group_key="acronym", cmap="magma", figsize=(14, 8),
                                              title="region x lipid (relative abundance, both axes clustered)")
plt.show()

# Blocks of regions sharing a block of high lipids are the lipidomic signature of anatomy. The white-matter regions light up on the sphingolipid columns, with no anatomy labels used in the sorting.

## Recap

- A **reference atlas** gives every section a shared 3D coordinate system plus region labels.
- **Registration** = affine (global) + nonlinear warp (local). ABBA does it; we ship its output.
- Coordinate → voxel (`⌊mm × 40⌋`) → region; a tilted cut shows up as a spread of AP levels.
- White matter straight from the atlas (`#cccccc`), and a myelin lipid traces it on its own.
- The region x lipid matrix (saved to `results/region_by_lipid.csv`) is the hinge for region-level comparisons and the gene join.

**Next (05):** compress the lipids into a handful of programs.